In [10]:
import pandas as pd
import numpy as np
import spacy
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from transformers import pipeline
import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to C:\Users\A
[nltk_data]     SINDHU\AppData\Roaming\nltk_data...


True

In [4]:
print("Loading data...")
# Load the actual CSV file into the DataFrame
df = pd.read_csv(r"C:\Users\A SINDHU\Downloads\fake_news_data.csv")

Loading data...


In [5]:
df.head()

,title,text,date,fake_or_factual
0,HOLLYWEIRD LIB SUSAN SARANDON Compares Muslim ...,There are two small problems with your analogy...,"Dec 30, 2015",Fake News
1,Elijah Cummings Called Trump Out To His Face ...,Buried in Trump s bonkers interview with New Y...,"April 6, 2017",Fake News
2,Hillary Clinton Says Half Her Cabinet Will Be...,"Women make up over 50 percent of this country,...","April 26, 2016",Fake News
3,Russian bombing of U.S.-backed forces being di...,WASHINGTON (Reuters) - U.S. Defense Secretary ...,"September 18, 2017",Factual News
4,Britain says window to restore Northern Irelan...,BELFAST (Reuters) - Northern Ireland s politic...,"September 4, 2017",Factual News


In [ ]:
# Drop missing values just to be safe

In [6]:
df = df.dropna(subset=['text', 'fake_or_factual'])

In [ ]:
# Rename the 'fake_or_factual' column to 'label' so it matches the rest of the code

In [7]:
df = df.rename(columns={'fake_or_factual': 'label'})

In [ ]:
# Map the text labels to the exact words the code expects

In [8]:
df['label'] = df['label'].map({'Factual News': 'Factual', 'Fake News': 'Fake'})

In [ ]:
# Optional: If the dataset is massive and takes too long to run, 
# uncomment the line below to test on just the first 500 rows:
# df = df.head(500) 

# ==========================================
# 2. NER & SENTIMENT ANALYSIS (spaCy & VADER)
# ==========================================

In [11]:
print("\nRunning NLP Feature Engineering (NER & Sentiment)...")
nlp = spacy.load("en_core_web_sm")
analyzer = SentimentIntensityAnalyzer()

def extract_entities(text):
    # Convert to string just in case there are weird data types in the CSV
    doc = nlp(str(text))
    return [ent.label_ for ent in doc.ents]


Running NLP Feature Engineering (NER & Sentiment)...


In [ ]:
# Extract Named Entities (NER)

In [12]:
df['entities'] = df['text'].apply(extract_entities)

# Extract Sentiment
df['sentiment_score'] = df['text'].apply(lambda x: analyzer.polarity_scores(str(x))['compound'])

print("\n--- Sample of Features ---")
display(df[['text', 'entities', 'sentiment_score']].head())


--- Sample of Features ---


,text,entities,sentiment_score
0,There are two small problems with your analogy...,"[CARDINAL, PERSON, NORP, PERSON, GPE, PERSON, ...",-0.3660
1,Buried in Trump s bonkers interview with New Y...,"[ORG, ORG, PERSON, PERSON, ORG, ORG, ORG, DATE...",-0.7973
2,"Women make up over 50 percent of this country,...","[PERCENT, ORG, PERCENT, PERCENT, ORG, PERCENT,...",0.9886
3,WASHINGTON (Reuters) - U.S. Defense Secretary ...,"[GPE, ORG, GPE, PERSON, DATE, GPE, GPE, GPE, C...",-0.3400
4,BELFAST (Reuters) - Northern Ireland s politic...,"[GPE, ORG, GPE, GPE, DATE, GPE, PERSON, GPE]",0.8590


In [ ]:
# ==========================================
# 3. TOPIC MODELING (TF-IDF + LDA)
# ==========================================

# Convert text to TF-IDF matrix (limited to top 1000 words to save memory)

In [13]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['text'].astype(str))

In [ ]:
# Apply LDA for Topic Modeling

In [14]:
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(tfidf_matrix)

LatentDirichletAllocation(n_components=5, random_state=42)

In [ ]:
# Display the top words per topic

In [15]:
feature_names = tfidf_vectorizer.get_feature_names_out()
print("\n--- Top Topics Extracted ---")
for topic_idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-6:-1]]
    print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")


--- Top Topics Extracted ---
Topic 1: hillary, pic, com, twitter, mccain
Topic 2: trump, said, president, clinton, state
Topic 3: military, syria, saudi, taiwan, county
Topic 4: putin, sanders, cruz, wikileaks, chicago
Topic 5: boiler, japanese, korea, acr, room


In [ ]:
# ==========================================
# 4. TRADITIONAL ML PIPELINE (Scikit-Learn)
# ==========================================


In [16]:
# Map labels to binary numbers for the math model: Factual = 0, Fake = 1
y = df['label'].map({"Factual": 0, "Fake": 1})

# Split the TF-IDF data
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, y, test_size=0.3, random_state=42)

# Train a Logistic Regression model
ml_model = LogisticRegression(random_state=42)
ml_model.fit(X_train, y_train)

LogisticRegression(random_state=42)

In [ ]:
# Evaluate using Accuracy, Precision, Recall, F1-Score

In [17]:
ml_predictions = ml_model.predict(X_test)
print("\n--- Traditional ML (TF-IDF + Logistic Regression) ---")
print(f"Accuracy: {accuracy_score(y_test, ml_predictions) * 100:.2f}%")
print(classification_report(y_test, ml_predictions, target_names=["Factual", "Fake"], zero_division=0))


--- Traditional ML (TF-IDF + Logistic Regression) ---
Accuracy: 88.33%
              precision    recall  f1-score   support

     Factual       0.88      0.91      0.89        32
        Fake       0.89      0.86      0.87        28

    accuracy                           0.88        60
   macro avg       0.88      0.88      0.88        60
weighted avg       0.88      0.88      0.88        60



In [ ]:
# ==========================================
# 5. HUGGING FACE TRANSFORMERS (LLM Integration)
# ==========================================
print("\nLoading Hugging Face Transformer Model...")

In [18]:
# Using a pre-trained fake news detection model from Hugging Face
transformer_pipeline = pipeline("text-classification", model="mrm8488/bert-tiny-finetuned-fake-news-detection")

def predict_with_llm(text):
    # Slice the text [:512] because BERT models can only read 512 tokens at a time
    result = transformer_pipeline(str(text)[:512])[0]
    return "Fake" if result['label'] == 'LABEL_1' else "Factual"

config.json:   0%|          | 0.00/705 [00:00<?, ?B/s]

C:\Anaconda\envs\nlp_course_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\A SINDHU\.cache\huggingface\hub\models--mrm8488--bert-tiny-finetuned-fake-news-detection. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falli

pytorch_model.bin:   0%|          | 0.00/17.6M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/360 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu


In [19]:
# Test the LLM on a brand new, unseen sentence
test_sentence = "Secret documents reveal the moon landing was filmed in a Hollywood basement."
llm_prediction = predict_with_llm(test_sentence)

print("\n--- Hugging Face Transformer Prediction ---")
print(f"Text: '{test_sentence}'")
print(f"Transformer Classification: {llm_prediction}")


--- Hugging Face Transformer Prediction ---
Text: 'Secret documents reveal the moon landing was filmed in a Hollywood basement.'
Transformer Classification: Fake
